# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring a dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via Croissant schema:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset-level metadata
meta = dataset.metadata
print("Dataset title:", meta.name)
print("Description:", meta.description)
print("Date published:", meta.datePublished)
print("Version:", meta.version)
if hasattr(meta, 'keywords'):
    print("Keywords:", ', '.join(meta.keywords))
print("License:", meta.license)


## 2. Data Overview
Review available record sets, their field/column `@id`s, and a sample of their schema.

We use the Croissant schema to discover the record sets, fields, and their unique `@id` identifiers.

In [ ]:
print("Available Record Sets:")
record_set_ids = []
for record_set in dataset.metadata.recordSet:
    print(f"  - Name: {record_set.name}")
    print(f"    @id: {record_set['@id']}")
    # List columns with @id
    if hasattr(record_set, 'column'):
        print("    Columns (@id):")
        for column in record_set.column:
            print(f"      - {column['@id']} (name: {column.name}, type: {column.dataType})")
    record_set_ids.append(record_set['@id'])
    print()
# For this dataset, let's select the main survey record set (usually the largest or most relevant)
main_record_set_id = record_set_ids[0] if record_set_ids else None


## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis using the `record_set` and column `@id`s as seen above.

We'll use the primary record set for exploration.

In [ ]:
# Load all records from each discovered record set
dfs = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f"Loaded {len(df)} rows from record set '{record_set_id}'")
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {e}")
        dfs[record_set_id] = pd.DataFrame()

# Use the main record set for demonstration
if main_record_set_id and main_record_set_id in dfs:
    main_df = dfs[main_record_set_id]
    print("First 5 columns:", main_df.columns[:5].tolist())
    display(main_df.head())
else:
    print("No main record set found or data unavailable.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate basic numeric filtering, normalization, and grouping using the fields found above.

We reference all columns by their `@id`.

In [ ]:
# Example: Find a numeric field to analyze, here we pick the first 'Float' or 'Integer'-typed field
numeric_field_id = None
group_field_id = None
for record_set in dataset.metadata.recordSet:
    if record_set['@id'] == main_record_set_id:
        for col in record_set.column:
            if col.dataType in ('Float', 'Integer') and (numeric_field_id is None):
                numeric_field_id = col['@id']
            if col.dataType == 'Text' and group_field_id is None:
                group_field_id = col['@id']
        break
if numeric_field_id:
    print(f"Numeric field selected: {numeric_field_id}")
else:
    print("No numeric field found.")

# If the DataFrame for the record set is available, run EDA
import numpy as np
if main_record_set_id in dfs and numeric_field_id and numeric_field_id in dfs[main_record_set_id]:
    df = dfs[main_record_set_id]
    # Cast to float, handle missing values
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    print(f"Numeric field stats: min={df[numeric_field_id].min()}, max={df[numeric_field_id].max()}, mean={df[numeric_field_id].mean()}")
    # Example filtering: keep values above median
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered rows where {numeric_field_id} > median ({threshold}): {len(filtered_df)} rows.")

    # Normalize this field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a string/text field if present and print group means
    if group_field_id is not None and group_field_id in filtered_df:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Group mean for {numeric_field_id} by {group_field_id}:")
        print(grouped.head())
else:
    print("EDA could not run: Main record set DataFrame or relevant field(s) missing.")

## 5. Visualization
Visualize the distribution of the numeric field and, if available, colored by a group field.

_All columns are referenced by their Croissant `@id`._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_record_set_id in dfs and numeric_field_id and numeric_field_id in dfs[main_record_set_id]:
    df = dfs[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df:
        # Boxplot by group
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization could not run due to missing fields or DataFrame.")

## 6. Conclusion

In this notebook, we demonstrated how to use `mlcroissant` to:
- Load and browse Croissant metadata and records using the Croissant schema URL (`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`).
- Discover all record sets and their field/Croissant `@id`s (ensuring reproducibility and schema clarity).
- Perform exploratory data analysis and visualizations on numeric fields referenced by `@id`, suitable for further statistical or machine learning work.

This dataset (FAIR² Mental Health Survey: Kilifi County, Kenya) offers a good example for learning highly FAIR, AI-ready dataset integration into analytical workflows. Further steps could include deeper domain-specific analysis or feature engineering as required for downstream tasks.